# Холодный старт: популярность и любимые жанры

Проверяем `GenrePopularityRecommender` на тех же cold-validation пользователях,
что в `01_eda.ipynb`. Сравниваем популярность за всё время и за 28 дней,
для каждого окна — обычную выдачу и жанровую с beta 0,5 / 1 / 2.

Без истории жанрового профиля нет: выдача должна совпасть с популярностью.
Поэтому результат для пользователей с 1–4 записями показываем отдельно;
именно на этой группе выбираем жанровую настройку. Для нулевой истории
сохраняем обычную популярность за всё время, выбранную по предыдущему validation.

Границы и правила релевантности не меняем. Финальный test не оценивается.
Текущий снимок и последний период уже исследовались ранее — это ограничение
независимости оценки остаётся в силе.

In [1]:
import sys
from pathlib import Path
import hashlib

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from IPython.display import display

from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate, ndcg_at_k
from app.training.anilist_genres import DEFAULT_GENRES_PATH, read_cache, load_item_genres
from app.recsys.popularity import PopularityRecommender, GenrePopularityRecommender

pd.set_option("display.float_format", "{:,.4f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
train_history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(train_history)
relevant = relevant_items(validation)
counts = train_history.user_id.value_counts()
cold = {u: items for u, items in relevant.items() if counts.get(u, 0) < 5}
no_history = {u: items for u, items in cold.items() if counts.get(u, 0) == 0}
short_history = {u: items for u, items in cold.items() if counts.get(u, 0) > 0}
groups = {"все cold": cold, "нет истории": no_history, "1–4 записи": short_history}
# Для профилей нужны только записи оцениваемых пользователей, строго из train.
cold_history = train_history[train_history.user_id.isin(cold)]
pd.Series({name: len(targets) for name, targets in groups.items()}, name="пользователей").to_frame()

,пользователей
все cold,616
нет истории,410
1–4 записи,206


## 1. Полнота жанров

Перед запуском из `ml/`:
```sh
.venv/bin/python -m app.training.anilist_genres --before 2026-09-02T00:34:23Z
```
Кэш `data/anilist/genres.json` хранит прогресс и дату получения метаданных.
Повторный запуск скачивает только ещё не обработанные ID; `--retry-missing`
повторяет запросы тайтлов, не вернувшихся в ответе.

Жанры используются по **названиям**, тайтлы — в **AniList ID**. Это не ID жанров
и тайтлов нашей БД. Метаданные получены сейчас, а не на дату отсечения: считаем
жанры относительно стабильными, но это допущение, не историческая реконструкция.
Хэш кэша позволяет отличить результаты с разными версиями метаданных.

In [2]:
cache = read_cache(DEFAULT_GENRES_PATH)
expected_ids = set(map(int, train_history.anime_id.unique()))
processed_ids = set(map(int, cache["items"]))
assert not (expected_ids - processed_ids), "Загрузка жанров ещё не завершена"
item_genres = load_item_genres()
known_ids = set(item_genres)
rows = [cache["items"][str(i)] for i in sorted(expected_ids)]
print("SHA256 кэша:", hashlib.sha256(DEFAULT_GENRES_PATH.read_bytes()).hexdigest())
print("Метаданные получены:", min(r["fetched_at"] for r in rows), "—", max(r["fetched_at"] for r in rows))
pd.Series({
    "тайтлов в полной train-history": len(expected_ids),
    "тайтлов с жанрами": len(expected_ids & known_ids),
    "недоступных тайтлов": sum(r["genres"] is None for r in rows),
    "тайтлов с пустыми жанрами": sum(r["genres"] == [] for r in rows),
    "покрытие записей обучения жанрами": training.anime_id.isin(known_ids).mean(),
    "покрытие короткой истории жанрами": cold_history.anime_id.isin(known_ids).mean(),
}, name="значение").to_frame()

SHA256 кэша: 6aaebf906577a399ecaf6755481d690c4923e8824ee3a137c3f4ff976443535a
Метаданные получены: 2026-11-11T15:23:11.168679+00:00 — 2026-11-11T20:10:49.743420+00:00


,значение
тайтлов в полной train-history,"21,241.0000"
тайтлов с жанрами,"18,396.0000"
недоступных тайтлов,1.0000
тайтлов с пустыми жанрами,"2,844.0000"
покрытие записей обучения жанрами,0.9949
покрытие короткой истории жанрами,1.0000


## 2. Кому можно построить профиль

Достаточно одной положительной записи с известным жанром. `dropped` и оценки
ниже 5 не формируют предпочтений, но все записи исключаются из выдачи.
Если положительных записей с жанрами нет, жанровая модель возвращает популярное.

In [3]:
probe = GenrePopularityRecommender(item_genres, window_days=None).fit(
    training, now=VALIDATION_START, history=cold_history)
profile_users = {u for u, pref in probe.preferences_.items() if pref.any()}
print(f"Профиль доступен {len(profile_users & set(short_history))} из {len(short_history)} пользователей с 1–4 записями")
print("Жанров в метаданных:", len({g for gs in item_genres.values() for g in gs}))
assert not (profile_users & set(no_history))

Профиль доступен 206 из 206 пользователей с 1–4 записями
Жанров в метаданных: 19


## 3. Сравнение на validation

Во всех вариантах одинаковые обучение, история и цели. Порядок вариантов при
равенстве NDCG предпочитает обычную популярность, затем меньший beta.
Пользователи без жанрового профиля не удаляются из оценки.
Coverage — число уникальных тайтлов в рекомендациях этой группы, не процент каталога.

In [4]:
records = []
recommendations = {}
for window in [None, 28]:
    window_name = "всё время" if window is None else "28 дней"
    plain = PopularityRecommender(window_days=window).fit(
        training, now=VALIDATION_START, history=cold_history)
    plain_recs = plain.recommend_many(cold, k=10)
    for beta in [0, 0.5, 1.0, 2.0]:
        if beta == 0:
            recs = plain_recs
        else:
            model = GenrePopularityRecommender(item_genres, window_days=window, beta=beta).fit(
                training, now=VALIDATION_START, history=cold_history)
            recs = model.recommend_many(cold, k=10)
            assert all(recs[u] == plain_recs[u] for u in no_history)
        key = (window_name, beta)
        recommendations[key] = recs
        for group, targets in groups.items():
            records.append({"окно": window_name, "beta": beta, "группа": group,
                            **evaluate(recs, targets, k=10)})
results = pd.DataFrame(records).set_index(["окно", "beta", "группа"])
display(results)
short_results = results.xs("1–4 записи", level="группа")
best = short_results["ndcg@10"].idxmax()
best_plain = short_results[short_results.index.get_level_values("beta") == 0]["ndcg@10"].idxmax()
print("Лучший вариант для 1–4 записей:", best)
print("Лучший baseline для 1–4 записей:", best_plain)

precision@10  recall@10  ndcg@10  coverage@10  \
окно      beta   группа                                                       
всё время 0.0000 все cold           0.1586     0.0302   0.1687           12   
                 нет истории        0.2029     0.0346   0.2147           10   
                 1–4 записи         0.0704     0.0215   0.0772           12   
          0.5000 все cold           0.1580     0.0299   0.1687           41   
                 нет истории        0.2029     0.0346   0.2147           10   
                 1–4 записи         0.0684     0.0205   0.0773           41   
          1.0000 все cold           0.1593     0.0298   0.1691           63   
                 нет истории        0.2029     0.0346   0.2147           10   
                 1–4 записи         0.0723     0.0203   0.0783           63   
          2.0000 все cold           0.1578     0.0296   0.1680           89   
                 нет истории        0.2029     0.0346   0.2147           10   
                 1–4 записи         0.0680     0.0197   0.0752           89   
28 дней   0.0000 все cold           0.0919     0.0375   0.1051           11   
                 нет истории        0.1073     0.0293   0.1173           10   
                 1–4 записи         0.0612     0.0540   0.0807           11   
          0.5000 все cold           0.0959     0.0422   0.1088           39   
                 нет истории        0.1073     0.0293   0.1173           10   
                 1–4 записи         0.0733     0.0678   0.0918           38   
          1.0000 все cold           0.0976     0.0462   0.1110           50   
                 нет истории        0.1073     0.0293   0.1173           10   
                 1–4 записи         0.0782     0.0800   0.0985           49   
          2.0000 все cold           0.0981     0.0439   0.1108           66   
                 нет истории        0.1073     0.0293   0.1173           10   
                 1–4 записи         0.0796     0.0729   0.0979           65   

                              users  
окно      beta   группа              
всё время 0.0000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
          0.5000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
          1.0000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
          2.0000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
28 дней   0.0000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
          0.5000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
          1.0000 все cold       616  
                 нет истории    410  
                 1–4 записи     206  
          2.0000 все cold       616  
                 нет истории    410  
                 1–4 записи     206

Лучший вариант для 1–4 записей: ('28 дней', np.float64(1.0))
Лучший baseline для 1–4 записей: ('28 дней', np.float64(0.0))


## 4. Проверка величины эффекта

Всего 206 пользователей с короткой историей — небольшой набор. Показываем
парную разницу NDCG по пользователям и bootstrap-интервал (2 000 повторов,
фиксированный seed). Вариант выбран на этой же выборке: интервал описательный,
не является независимым доказательством статистической значимости.

In [5]:
users = sorted(short_history)
deltas = np.array([
    ndcg_at_k(recommendations[best][u], short_history[u], 10)
    - ndcg_at_k(recommendations[best_plain][u], short_history[u], 10)
    for u in users
])
rng = np.random.default_rng(42)
bootstrap = rng.choice(deltas, size=(2000, len(deltas)), replace=True).mean(axis=1)
print(f"Средняя разница NDCG@10: {deltas.mean():+.6f}")
print("Описательный 95% bootstrap-интервал:", np.quantile(bootstrap, [0.025, 0.975]))
print("Пользователей: улучшение / без изменения / ухудшение:",
      int((deltas > 0).sum()), int((deltas == 0).sum()), int((deltas < 0).sum()))

Средняя разница NDCG@10: +0.017786
Описательный 95% bootstrap-интервал: [0.00547517 0.03076616]
Пользователей: улучшение / без изменения / ухудшение: 46 131 29


## Вывод

- Кэш покрывает 21 241 тайтл полной train-history: 18 396 с жанрами, 2 844 без
  жанров, один недоступен и после повторного запроса. Жанры покрывают 99,49%
  записей обучения и 100% короткой истории; профиль есть у всех 206 пользователей.
- **Без истории (410 пользователей):** популярность за всё время — NDCG@10
  0,2147 против 0,1173 у окна 28 дней. Жанры не меняют выдачу, что проверено assert.
- **1–4 записи (206 пользователей):** лучшее из проверенных сочетаний — окно
  28 дней, beta=1: NDCG@10 0,0985 против 0,0807 у лучшей обычной популярности
  (тоже 28 дней). Прирост около 22%, recall@10 0,0800 против 0,0540,
  coverage@10 — 49 тайтлов против 11.
- По NDCG улучшение у 46 пользователей, ухудшение у 29, без изменения у 131.
  Средняя парная разница +0,0178, описательный bootstrap-интервал
  [0,0055; 0,0308]. Подбор beta на той же выборке ограничивает вывод:
  это предварительный результат validation, не независимая проверка гипотезы.
- **Предварительная схема для интеграции:** без истории — популярность за всё
  время; короткая положительная история с жанрами — жанровая популярность
  за 28 дней с beta=1. Сценарий короткой истории без доступного профиля
  в этой выборке не представлен; базовый fallback — обычная популярность.
- Общее среднее по cold скрывает разницу этих сценариев, поэтому единое окно
  для всех новичков выбирать по нему не следует.
- Следующий этап — простой ALS и сравнение на warm-validation. Финальный test
  оставляем для оценки после выбора моделей. Значения метрик и настройки
  в этом разделе не являются результатами тестового периода.
